# 99 · End-to-end orchestrator
Thin wrapper: raw CSV -> DuckDB -> clean -> bin -> train -> calibrate -> evaluate -> threshold -> reports. All logic lives in `src/`; this notebook only calls `run_all` and displays the headline outputs. It also writes `reports/model_card.md`, `reports/validation_memo.md`, `reports/threshold_tradeoff.csv` and the figures.

In [ ]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.utils.config import load_config, set_seeds
cfg = load_config(); set_seeds(cfg["seed"])
pd.set_option("display.float_format", "{:.4f}".format); pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
print("config seed:", cfg["seed"], "| raw file:", cfg["paths"]["raw_csv"])

In [ ]:
from src.pipeline import run_all
res = run_all(cfg, rebuild=False, make_reports=True)

## Results table (calibrated PD)

In [ ]:
p = res["perf"]; p[(p.pd_type=="calibrated") & (p["sample"]!="train (in-sample)")][["model","sample","n","default_rate","mean_pd","auc","gini","ks","brier"]]

## Champion calibration (2015)

In [ ]:
res["cal"]["champion_logit"]["summary"]

## Threshold recommendation (illustrative; assumed appetite)

In [ ]:
r = res["rec"]; print(f"approve {r.approval_rate:.0%} | PD cutoff {r.pd_cutoff:.3f} | approved-book EL rate {r.el_rate_model:.2%} | appetite {cfg['decision']['el_appetite']:.1%}")
res["trade"]

## Score PSI (distribution shift, not performance)

In [ ]:
res["psi"]["score"]

In [ ]:
print(sorted(p.name for p in (ROOT/cfg["paths"]["reports_dir"]).glob("*")), sorted(p.name for p in (ROOT/cfg["paths"]["figures_dir"]).glob("*.png")))